# Week 9 — Tree-Based Methods & Gradient Boosting
## Notebook 2: Splitting Criteria — Gini, Entropy & Variance Reduction

**Difficulty:** ⭐⭐⭐ (Intermediate) | **Estimated Time:** 2.5 hours

---

### Why This Matters

The CART algorithm (Notebook 1) must **score every candidate split** to decide which one is best. The scoring function — called the **splitting criterion** — is the mathematical heart of the algorithm. Choose the wrong criterion and the tree might cut on the wrong features or the wrong thresholds.

In this notebook you will:
- Derive and implement **Gini impurity**, **Entropy/Information Gain**, and **Variance Reduction** from first principles
- Visualise how each criterion behaves across all possible class distributions
- See which features are most *informative* according to each measure
- Compare Gini vs Entropy on a real classification task and analyse when they agree (usually) and when they differ (rarely)
- Understand the full split-candidate evaluation loop that sklearn executes at each node

---

### Real-World Analogy First

Imagine sorting a **jumbled bag of M&Ms** into groups:

- A bag containing **100 red M&Ms** is perfectly *pure* — you already know what every M&M is. No sorting needed.
- A bag with **equal numbers of red, blue, green, yellow, and orange** is maximally *mixed* — you can't guess the colour of a random one better than chance.

**Gini impurity** measures how mixed a bag is: pull out a random M&M, then pull another — what's the probability they are **different colours**? Pure bag = 0 probability (Gini = 0). Totally mixed = high probability (Gini near 1).

**Information Gain** (Entropy-based) asks: how surprised would you be by a random M&M's colour? Pure bag = 0 surprise (you already know). Mixed bag = maximum surprise.

The CART algorithm at each node asks: **which question (feature ≤ threshold) reduces the mixing the most?** That question becomes the split.

---

### Roadmap

```
1. Dataset creation
2. Implement Gini, Entropy, Variance Reduction from scratch
3. Plot impurity curves as functions of class probability
4. Impurity landscape: Gini and Entropy for every threshold on sqft
5. Criterion comparison: Gini vs Entropy vs log_loss
6. Variance reduction for regression
7. Class probability scatter and decision boundaries per criterion
8. Feature ranking by split gain
```

In [ ]:
# ── Imports & global settings ─────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Reproducibility and aesthetics
np.random.seed(42)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print('Libraries loaded.')

## Section 1 — Dataset

We generate **1 000 houses** with the same 6 features as Notebook 1. We then create **two label sets**:

1. **Binary (`above_median`)**: 0 = below median, 1 = above median — used for Gini/Entropy comparisons.
2. **3-class (`price_tier`)**: low / mid / high price tier — demonstrates multi-class impurity measures.

In [ ]:
# ── 1. Generate house price dataset (1 000 samples) ───────────────────────────
np.random.seed(42)
n = 1000

sqft         = np.random.uniform(600, 4000, n)
bedrooms     = np.random.randint(1, 7, n).astype(float)
bathrooms    = np.random.randint(1, 5, n).astype(float)
age          = np.random.uniform(0, 60, n)
school_score = np.random.uniform(1, 10, n)
dist_center  = np.random.uniform(1, 30, n)

price = (  80 * sqft
         + 15000 * bedrooms
         + 10000 * bathrooms
         -  1500 * age
         +  8000 * school_score
         -  3000 * dist_center
         + np.random.normal(0, 25000, n)
        )
price = np.clip(price, 50000, None)

# Binary label: above vs below median
median_price = np.median(price)
above_median = (price > median_price).astype(int)

# 3-class label: low (0) / mid (1) / high (2)
q33, q67 = np.percentile(price, [33, 67])
price_tier = np.digitize(price, bins=[q33, q67])   # 0, 1, 2

feature_names = ['sqft', 'bedrooms', 'bathrooms', 'age', 'school_score', 'dist_center']

df = pd.DataFrame(dict(
    sqft=sqft, bedrooms=bedrooms, bathrooms=bathrooms,
    age=age, school_score=school_score, dist_center=dist_center,
    price=price, above_median=above_median, price_tier=price_tier
))

X = df[feature_names].values
y_bin  = df['above_median'].values
y_3cls = df['price_tier'].values

print(f'Dataset shape   : {df.shape}')
print(f'Median price    : ${median_price:,.0f}')
print(f'Binary balance  : {y_bin.mean()*100:.1f}% above median')
print(f'3-class counts  : {np.bincount(y_3cls)}  (low/mid/high)')

## Section 2 — Gini Impurity: Derivation and Implementation

### What is Gini Impurity?

Given a node with class distribution $p_1, p_2, \ldots, p_K$ (where $\sum_k p_k = 1$):

$$\text{Gini}(t) = 1 - \sum_{k=1}^{K} p_k^2$$

**Interpretation**: pick two samples from the node at random with replacement. Gini is the probability they belong to **different classes**. A pure node has $p_1=1$ (everyone is the same class), so Gini = $1 - 1^2 = 0$. Maximum impurity for $K$ balanced classes = $1 - 1/K$.

**For binary classification** (classes 0 and 1, where $p$ = fraction of class 1):

$$\text{Gini}_\text{binary} = 1 - p^2 - (1-p)^2 = 2p(1-p)$$

This is maximised at $p = 0.5$ (value = 0.5) and zero at $p = 0$ or $p = 1$ (pure nodes).

### Split Gain

When we split a parent node (n samples) into left ($n_L$) and right ($n_R$) children:

$$\text{Gain} = \text{Gini}(\text{parent}) - \left(\frac{n_L}{n}\cdot\text{Gini}(\text{left}) + \frac{n_R}{n}\cdot\text{Gini}(\text{right})\right)$$

We want to **maximise** this gain — find the split that most reduces impurity.

In [ ]:
# ── 2. Impurity functions — implemented from scratch ──────────────────────────

def gini(y):
    """
    Gini impurity for integer class labels y.
    Returns 0.0 for a pure node, up to (1 - 1/K) for K balanced classes.
    """
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / len(y)              # class probability vector
    return 1.0 - np.sum(p ** 2)      # 1 - Σ p_k^2


def entropy(y):
    """
    Shannon entropy (bits) for integer class labels y.
    Returns 0.0 for a pure node, log2(K) for K balanced classes.
    """
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    return -np.sum(p * np.log2(p + 1e-10))  # -Σ p_k log2(p_k), +eps avoids log(0)


def information_gain(y, y_left, y_right):
    """
    Information gain = parent entropy - weighted child entropy.
    """
    n, n_l, n_r = len(y), len(y_left), len(y_right)
    return entropy(y) - (n_l / n) * entropy(y_left) - (n_r / n) * entropy(y_right)


def gini_gain(y, y_left, y_right):
    """
    Gini gain = parent Gini - weighted child Gini.
    """
    n, n_l, n_r = len(y), len(y_left), len(y_right)
    return gini(y) - (n_l / n) * gini(y_left) - (n_r / n) * gini(y_right)


def variance_reduction(y, y_left, y_right):
    """
    Variance reduction for regression: parent variance - weighted child variance.
    Equivalent to choosing the split that minimises weighted MSE.
    """
    n, n_l, n_r = len(y), len(y_left), len(y_right)
    return np.var(y) - (n_l / n) * np.var(y_left) - (n_r / n) * np.var(y_right)


# ── Sanity checks ──────────────────────────────────────────────────────────────
pure   = np.array([1, 1, 1, 1, 1])          # all class 1 → Gini=0, Entropy=0
mixed  = np.array([0, 0, 1, 1])             # 50/50 binary → Gini=0.5, Entropy=1.0
mixed3 = np.array([0, 1, 2, 0, 1, 2])       # 3 balanced classes

print('=== Sanity checks ===')
print(f'Pure node  → Gini = {gini(pure):.4f}  (expected 0.0)')
print(f'Pure node  → Entropy = {entropy(pure):.4f}  (expected 0.0)')
print(f'50/50 binary → Gini = {gini(mixed):.4f}  (expected 0.5)')
print(f'50/50 binary → Entropy = {entropy(mixed):.4f}  (expected 1.0)')
print(f'3 balanced classes → Gini = {gini(mixed3):.4f}  (expected {2/3:.4f})')
print(f'3 balanced classes → Entropy = {entropy(mixed3):.4f}  (expected {np.log2(3):.4f})')

In [ ]:
# ── 3. Plot Gini and Entropy as functions of p (binary classification) ─────────
# For binary class with fraction p of class 1:
#   Gini(p)   = 2p(1-p)
#   Entropy(p) = -p log2(p) - (1-p) log2(1-p)
# Both are 0 at p=0 and p=1 (pure), maximum at p=0.5 (most mixed).

p_vals = np.linspace(0.001, 0.999, 500)    # avoid log(0)
gini_vals    = 2 * p_vals * (1 - p_vals)   # closed-form binary Gini
entropy_vals = -(p_vals * np.log2(p_vals)
                 + (1 - p_vals) * np.log2(1 - p_vals))
# Misclassification error (for comparison)
misclass_vals = 1 - np.maximum(p_vals, 1 - p_vals)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left panel: all three criteria on same scale ──────────────────────────────
axes[0].plot(p_vals, gini_vals,    color='steelblue', lw=2.5, label='Gini impurity')
axes[0].plot(p_vals, entropy_vals / 2, color='darkorange', lw=2.5,
             label='Entropy / 2  (scaled to [0, 0.5])')
axes[0].plot(p_vals, misclass_vals, color='seagreen', lw=2.5, ls='--',
             label='Misclassification error')

# Annotate p=0.1 — show how entropy penalises impurity more than Gini
p_annot = 0.1
g_ann = 2 * p_annot * (1 - p_annot)
e_ann = -(p_annot * np.log2(p_annot) + (1 - p_annot) * np.log2(1 - p_annot)) / 2
axes[0].annotate(f'p=0.1\nGini={g_ann:.2f}\nEntropy/2={e_ann:.2f}',
                 xy=(p_annot, g_ann), xytext=(0.18, 0.25),
                 arrowprops=dict(arrowstyle='->', color='black'),
                 fontsize=9, color='black')

axes[0].set_xlabel('Class 1 probability  p')
axes[0].set_ylabel('Impurity measure')
axes[0].set_title('Impurity Curves for Binary Classification\n'
                  'All measures: 0 at pure nodes, maximum at p=0.5')
axes[0].legend(fontsize=9)

# ── Right panel: Gini and Entropy on their natural scales ────────────────────
axes[1].plot(p_vals, gini_vals,    color='steelblue', lw=2.5, label='Gini  (max=0.5 at p=0.5)')
axes[1].plot(p_vals, entropy_vals, color='darkorange', lw=2.5, label='Entropy  (max=1.0 at p=0.5)')

# Mark key values
axes[1].axvline(0.5, color='gray', ls=':', lw=1.5)
axes[1].axhline(0.5, color='steelblue', ls=':', lw=1.5, alpha=0.5)
axes[1].axhline(1.0, color='darkorange', ls=':', lw=1.5, alpha=0.5)
axes[1].set_xlabel('Class 1 probability  p')
axes[1].set_ylabel('Impurity measure (natural scale)')
axes[1].set_title('Gini vs Entropy — Natural Scales\n'
                  'Entropy grows more steeply near p=0.5')
axes[1].legend(fontsize=9)

plt.suptitle('Splitting Criteria as Functions of Class Probability',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key property: Entropy is more sensitive to changes near p=0.5')
print('This means entropy sometimes prefers more balanced splits.')

In [ ]:
# ── 3b. Multi-class Gini: 3 classes at various proportions ────────────────────
# For 3 classes [p, q, 1-p-q]: show Gini as heatmap over p-q space

p_grid = np.linspace(0, 1, 200)
q_grid = np.linspace(0, 1, 200)
PP, QQ = np.meshgrid(p_grid, q_grid)
RR = 1 - PP - QQ    # third class probability

# Only valid where all probabilities are in [0,1]
valid = (RR >= 0) & (RR <= 1)
GINI_3 = np.where(valid, 1 - PP**2 - QQ**2 - RR**2, np.nan)

fig, ax = plt.subplots(figsize=(7, 6))
cf = ax.contourf(PP, QQ, GINI_3, levels=20, cmap='YlOrRd')
plt.colorbar(cf, ax=ax, label='Gini Impurity')
ax.contour(PP, QQ, GINI_3, levels=10, colors='black', linewidths=0.4, alpha=0.5)

# Mark corners (pure nodes)
for (px, qx, label) in [(1,0,'p=1 (pure cls 0)'), (0,1,'p=0,q=1 (pure cls 1)'),
                          (0,0,'all cls 2')]:
    ax.scatter(px, qx, color='white', s=100, zorder=5, edgecolors='black')
    ax.annotate(label, (px, qx), textcoords='offset points', xytext=(8, 4), fontsize=8)

ax.set_xlabel('p₁  (proportion of class 0)')
ax.set_ylabel('p₂  (proportion of class 1)')
ax.set_title('3-Class Gini Impurity Heatmap\n'
             'Corners = pure (Gini=0); centre = maximum mixing')
plt.tight_layout()
plt.show()

## Section 3 — Impurity Landscape for a Single Feature

Before the CART algorithm picks the best split, it evaluates **every possible threshold** on each feature. Let's visualise what this looks like for `sqft`:

For each threshold $t$:
1. Split dataset: left = `{sqft ≤ t}`, right = `{sqft > t}`
2. Compute weighted Gini and weighted Entropy after the split
3. The threshold that minimises the weighted impurity (or equivalently, maximises the gain) is selected

The impurity *landscape* is the curve of weighted impurity across all thresholds. The minimum of this curve is where the algorithm cuts.

In [ ]:
# ── 4. Impurity landscape: Gini and Entropy at every sqft threshold ────────────

# Sort by sqft and evaluate each unique value as a candidate threshold
sqft_vals   = X[:, 0]          # feature 0 = sqft
thresholds  = np.unique(sqft_vals)

weighted_gini_vals    = []
weighted_entropy_vals = []
gini_gain_vals        = []
info_gain_vals        = []

parent_gini    = gini(y_bin)
parent_entropy = entropy(y_bin)
n_total        = len(y_bin)

for t in thresholds:
    left_mask  = sqft_vals <= t
    right_mask = ~left_mask
    n_l, n_r   = left_mask.sum(), right_mask.sum()

    if n_l == 0 or n_r == 0:
        weighted_gini_vals.append(np.nan)
        weighted_entropy_vals.append(np.nan)
        gini_gain_vals.append(np.nan)
        info_gain_vals.append(np.nan)
        continue

    wg = (n_l / n_total) * gini(y_bin[left_mask])    + (n_r / n_total) * gini(y_bin[right_mask])
    we = (n_l / n_total) * entropy(y_bin[left_mask]) + (n_r / n_total) * entropy(y_bin[right_mask])

    weighted_gini_vals.append(wg)
    weighted_entropy_vals.append(we)
    gini_gain_vals.append(parent_gini - wg)
    info_gain_vals.append(parent_entropy - we)

weighted_gini_vals    = np.array(weighted_gini_vals)
weighted_entropy_vals = np.array(weighted_entropy_vals)
gini_gain_vals        = np.array(gini_gain_vals)
info_gain_vals        = np.array(info_gain_vals)

# Best thresholds
best_t_gini    = thresholds[np.nanargmin(weighted_gini_vals)]
best_t_entropy = thresholds[np.nanargmin(weighted_entropy_vals)]

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# ── Top: weighted impurity after split ───────────────────────────────────────
axes[0].plot(thresholds, weighted_gini_vals,    lw=1.8, color='steelblue', label='Weighted Gini after split')
axes[0].plot(thresholds, weighted_entropy_vals, lw=1.8, color='darkorange', label='Weighted Entropy after split')
axes[0].axvline(best_t_gini,    color='steelblue', ls='--', lw=1.5,
                label=f'Best Gini threshold: sqft={best_t_gini:.0f}')
axes[0].axvline(best_t_entropy, color='darkorange', ls='--', lw=1.5,
                label=f'Best Entropy threshold: sqft={best_t_entropy:.0f}')
axes[0].axhline(parent_gini,    color='steelblue', ls=':', lw=1, alpha=0.5, label='Parent Gini (no split)')
axes[0].set_xlabel('sqft threshold')
axes[0].set_ylabel('Weighted impurity after split')
axes[0].set_title('Impurity Landscape: Weighted Gini & Entropy at Every sqft Threshold\n'
                  'CART picks the threshold that MINIMISES this curve')
axes[0].legend(fontsize=9)

# ── Bottom: gain (= parent impurity - weighted child impurity) ─────────────
axes[1].plot(thresholds, gini_gain_vals,  lw=1.8, color='steelblue', label='Gini Gain')
axes[1].plot(thresholds, info_gain_vals,  lw=1.8, color='darkorange', label='Information Gain')
axes[1].axvline(best_t_gini, color='steelblue', ls='--', lw=1.5)
axes[1].axhline(0, color='black', lw=0.8, ls=':')
axes[1].set_xlabel('sqft threshold')
axes[1].set_ylabel('Gain (impurity reduction)')
axes[1].set_title('Gain Landscape — CART picks the threshold that MAXIMISES gain')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'Best Gini split:    sqft ≤ {best_t_gini:.1f}  (Gini gain = {np.nanmax(gini_gain_vals):.4f})')
print(f'Best Entropy split: sqft ≤ {best_t_entropy:.1f}  (Info gain = {np.nanmax(info_gain_vals):.4f})')

## Section 4 — Gini vs Entropy vs log_loss: Practical Comparison

sklearn supports three classification criteria:

| Criterion | Formula | Notes |
|---|---|---|
| `gini` | $1 - \sum p_k^2$ | Default; fast (no log) |
| `entropy` | $-\sum p_k \log_2 p_k$ | Slightly more sensitive near 50/50 |
| `log_loss` | Same as entropy in sklearn | Alias for entropy |

In practice they almost always select the **same splits** and produce the **same accuracy**. The differences are subtle and usually only matter in edge cases with highly imbalanced classes.

In [ ]:
# ── 5. Criterion comparison: Gini vs Entropy vs log_loss ──────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_bin, test_size=0.25, random_state=42, stratify=y_bin
)

criteria = ['gini', 'entropy', 'log_loss']
results  = []

for crit in criteria:
    for depth in [3, 5, 7, None]:
        clf = DecisionTreeClassifier(criterion=crit, max_depth=depth, random_state=42)
        clf.fit(X_tr, y_tr)
        train_acc = accuracy_score(y_tr, clf.predict(X_tr))
        test_acc  = accuracy_score(y_te, clf.predict(X_te))
        results.append({
            'criterion': crit,
            'max_depth': str(depth) if depth else 'None',
            'train_acc': train_acc,
            'test_acc':  test_acc,
            'n_leaves':  clf.get_n_leaves()
        })

results_df = pd.DataFrame(results)

# Pivot for clean comparison
pivot = results_df.pivot_table(
    index='max_depth', columns='criterion',
    values='test_acc', aggfunc='first'
)
print('=== Test Accuracy by Criterion and Depth ===')
print((pivot * 100).round(2).to_string(), '  (% accuracy)')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for crit, color in zip(criteria, ['steelblue', 'darkorange', 'seagreen']):
    sub = results_df[results_df['criterion'] == crit]
    x_pos = range(len(sub))
    axes[0].plot(x_pos, sub['test_acc'] * 100, 'o-', lw=2, color=color, label=crit)

axes[0].set_xticks(range(4))
axes[0].set_xticklabels(['depth=3', 'depth=5', 'depth=7', 'depth=None'])
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Test Accuracy: Gini vs Entropy vs log_loss')
axes[0].legend()

# Difference plot (Gini minus Entropy) — should be near 0
gini_acc    = results_df[results_df['criterion'] == 'gini']['test_acc'].values
entropy_acc = results_df[results_df['criterion'] == 'entropy']['test_acc'].values
diffs = (gini_acc - entropy_acc) * 100
axes[1].bar(range(4), diffs, color=['green' if d >= 0 else 'red' for d in diffs])
axes[1].axhline(0, color='black', lw=1)
axes[1].set_xticks(range(4))
axes[1].set_xticklabels(['depth=3', 'depth=5', 'depth=7', 'depth=None'])
axes[1].set_ylabel('Accuracy difference (Gini − Entropy, pp)')
axes[1].set_title('Accuracy Difference: Gini minus Entropy\n'
                  'Usually very small — both criteria are nearly equivalent')

plt.tight_layout()
plt.show()

In [ ]:
# ── 5b. Do Gini and Entropy trees learn the same structure? ───────────────────
# Compare the first split (root node) of each tree at depth 3

for crit in ['gini', 'entropy']:
    clf = DecisionTreeClassifier(criterion=crit, max_depth=3, random_state=42)
    clf.fit(X_tr, y_tr)
    root_feat  = feature_names[clf.tree_.feature[0]]
    root_thresh = clf.tree_.threshold[0]
    print(f'{crit:8s}  root split: {root_feat} ≤ {root_thresh:.2f}')

print()
print('Note: Gini and Entropy usually produce identical splits on this dataset.')
print('Differences arise mainly on multi-class problems or near-tie situations.')

# Visualise both trees side by side (depth 3)
fig, axes = plt.subplots(1, 2, figsize=(22, 7))
for i, crit in enumerate(['gini', 'entropy']):
    clf = DecisionTreeClassifier(criterion=crit, max_depth=3, random_state=42)
    clf.fit(X_tr, y_tr)
    plot_tree(clf, feature_names=feature_names,
              class_names=['Below', 'Above'],
              filled=True, rounded=True, fontsize=8, ax=axes[i])
    acc = accuracy_score(y_te, clf.predict(X_te))
    axes[i].set_title(f'Criterion = {crit}  |  Test acc = {acc*100:.1f}%',
                      fontsize=11, fontweight='bold')

plt.suptitle('Tree Structure Comparison: Gini vs Entropy (depth=3)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 5 — Variance Reduction for Regression

For **regression** targets (continuous values), Gini and Entropy are undefined — there are no class probabilities.

Instead, CART minimises **weighted variance** (equivalently, weighted MSE):

$$\Delta\text{Var} = \text{Var}(y) - \frac{n_L}{n}\text{Var}(y_L) - \frac{n_R}{n}\text{Var}(y_R)$$

Why variance? Because the leaf prediction is the **mean** of all samples in the leaf. The prediction error for a sample in a leaf is $(y_i - \bar{y}_\text{leaf})$, and the average squared error is exactly the variance of $y$ within the leaf.

So minimising weighted variance = minimising weighted MSE = minimising the total squared error of all leaf predictions.

In [ ]:
# ── 6. Variance reduction for regression: sqft threshold landscape ─────────────
y_reg = df['price'].values

# Compute variance reduction at every sqft threshold
var_red_vals       = []
weighted_var_vals  = []
parent_var         = np.var(y_reg)

for t in thresholds:                              # re-use thresholds from section 3
    left_mask  = sqft_vals <= t
    right_mask = ~left_mask
    n_l, n_r   = left_mask.sum(), right_mask.sum()

    if n_l == 0 or n_r == 0:
        var_red_vals.append(np.nan)
        weighted_var_vals.append(np.nan)
        continue

    wv = ((n_l / n_total) * np.var(y_reg[left_mask])
          + (n_r / n_total) * np.var(y_reg[right_mask]))
    var_red_vals.append(parent_var - wv)  # reduction
    weighted_var_vals.append(wv)

var_red_vals      = np.array(var_red_vals)
weighted_var_vals = np.array(weighted_var_vals)
best_t_var = thresholds[np.nanargmin(weighted_var_vals)]

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Distribution of price with the best-split line
axes[0].hist(sqft_vals[sqft_vals <= best_t_var], bins=40,
             color='steelblue', alpha=0.7, label=f'Left leaf (sqft ≤ {best_t_var:.0f})')
axes[0].hist(sqft_vals[sqft_vals > best_t_var], bins=40,
             color='darkorange', alpha=0.7, label=f'Right leaf (sqft > {best_t_var:.0f})')
axes[0].axvline(best_t_var, color='black', lw=2, ls='--', label=f'Best split = {best_t_var:.0f}')
axes[0].set_xlabel('sqft')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Optimal sqft Split for Variance Reduction: sqft ≤ {best_t_var:.0f}')
axes[0].legend()

# Variance reduction landscape
axes[1].plot(thresholds, var_red_vals / 1e10, lw=2, color='seagreen', label='Variance reduction')
axes[1].axvline(best_t_var, color='black', lw=2, ls='--',
                label=f'Maximum at sqft = {best_t_var:.0f}')
axes[1].set_xlabel('sqft threshold')
axes[1].set_ylabel('Variance reduction (×10¹⁰)')
axes[1].set_title('Variance Reduction Landscape for sqft Feature\n'
                  'CART picks the threshold at the peak of this curve')
axes[1].legend()

plt.tight_layout()
plt.show()

# Compare mean price in each half
left_prices  = y_reg[sqft_vals <= best_t_var]
right_prices = y_reg[sqft_vals > best_t_var]
print(f'Best regression split: sqft ≤ {best_t_var:.0f}')
print(f'Left  leaf: n={len(left_prices):4d}, mean=${left_prices.mean()/1000:6.1f}k,  std=${left_prices.std()/1000:5.1f}k')
print(f'Right leaf: n={len(right_prices):4d}, mean=${right_prices.mean()/1000:6.1f}k,  std=${right_prices.std()/1000:5.1f}k')
print(f'Parent std: ${np.std(y_reg)/1000:.1f}k  →  splits reduce within-group std significantly.')

In [ ]:
# ── 7. Class probability scatter — visualise splits per criterion ──────────────
# For 2D view: sqft vs school_score, colour = class label
X2 = X[:, [0, 4]]   # sqft (col 0), school_score (col 4)
feat2_names = ['sqft', 'school_score']

# Build meshgrid for background
x1r = np.linspace(X2[:, 0].min(), X2[:, 0].max(), 300)
x2r = np.linspace(X2[:, 1].min(), X2[:, 1].max(), 300)
XX1, XX2 = np.meshgrid(x1r, x2r)
grid_pts  = np.c_[XX1.ravel(), XX2.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
palette = {0: '#2E86C1', 1: '#E67E22'}   # blue = below, orange = above median

for ax, crit in zip(axes, ['gini', 'entropy', 'log_loss']):
    clf2d = DecisionTreeClassifier(criterion=crit, max_depth=4, random_state=42)
    clf2d.fit(X2, y_bin)

    Z = clf2d.predict(grid_pts).reshape(XX1.shape)
    ax.contourf(XX1, XX2, Z, levels=[-0.5, 0.5, 1.5],
                colors=['#AED6F1', '#FAD7A0'], alpha=0.6)
    ax.contour(XX1, XX2, Z, levels=[0.5], colors='black', linewidths=1.5)

    point_colors = [palette[c] for c in y_bin]
    ax.scatter(X2[:, 0], X2[:, 1], c=point_colors,
               s=12, alpha=0.7, edgecolors='none')

    acc = accuracy_score(y_bin, clf2d.predict(X2))
    ax.set_title(f'criterion = {crit}\n'
                 f'Accuracy = {acc*100:.1f}% (training)',
                 fontweight='bold')
    ax.set_xlabel('sqft')
    ax.set_ylabel('school_score')

patch_b = mpatches.Patch(color='#AED6F1', label='Below median')
patch_a = mpatches.Patch(color='#FAD7A0', label='Above median')
axes[-1].legend(handles=[patch_b, patch_a], loc='lower right')

plt.suptitle('Decision Boundaries by Splitting Criterion (depth=4)\n'
             'Boundaries are nearly identical — criteria agree on most splits',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 6 — Feature Ranking by Split Gain

For each feature we find its **single best split** and compute the resulting Gini gain. This tells us:
- Which feature is most informative overall (best possible first split)
- Whether different features dominate for Gini vs Information Gain

This is related to (but not identical to) sklearn's `feature_importances_`, which aggregates gain across all splits in the fitted tree — not just the first one.

In [ ]:
# ── 8. Feature ranking by best single-split Gini gain ─────────────────────────

def best_split_gain(X_col, y, criterion_fn, gain_fn):
    """
    Find the threshold that maximises gain for a single feature column.
    Returns (best_threshold, best_gain, all_gains, thresholds).
    """
    thresholds  = np.unique(X_col)
    gains       = []
    parent_imp  = criterion_fn(y)
    n_total     = len(y)

    for t in thresholds:
        left_mask  = X_col <= t
        right_mask = ~left_mask
        n_l, n_r   = left_mask.sum(), right_mask.sum()
        if n_l == 0 or n_r == 0:
            gains.append(0.0)
            continue
        g = gain_fn(y, y[left_mask], y[right_mask])
        gains.append(g)

    gains = np.array(gains)
    best_idx = np.argmax(gains)
    return thresholds[best_idx], gains[best_idx], gains, thresholds


# Build summary table: best Gini gain and best Info Gain per feature
rows = []
for j, fname in enumerate(feature_names):
    bt_g, bg, _, _   = best_split_gain(X[:, j], y_bin, gini,    gini_gain)
    bt_e, be, _, _   = best_split_gain(X[:, j], y_bin, entropy, information_gain)
    rows.append({
        'Feature':            fname,
        'Best Gini Gain':     bg,
        'Best Gini Threshold':bt_g,
        'Best Info Gain':     be,
        'Best IG Threshold':  bt_e,
    })

gain_df = pd.DataFrame(rows).sort_values('Best Gini Gain', ascending=False)
print('=== Feature Ranking by Best Single-Split Gain ===')
print(gain_df.to_string(index=False, float_format='{:.5f}'.format))

In [ ]:
# ── 8b. Visualise the feature ranking ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sort for plotting
gain_df_sorted_g = gain_df.sort_values('Best Gini Gain')
gain_df_sorted_e = gain_df.sort_values('Best Info Gain')

axes[0].barh(gain_df_sorted_g['Feature'], gain_df_sorted_g['Best Gini Gain'],
             color='steelblue', edgecolor='white')
axes[0].set_xlabel('Best Gini Gain')
axes[0].set_title('Feature Ranking — Best Single-Split\nGini Gain')

axes[1].barh(gain_df_sorted_e['Feature'], gain_df_sorted_e['Best Info Gain'],
             color='darkorange', edgecolor='white')
axes[1].set_xlabel('Best Information Gain (bits)')
axes[1].set_title('Feature Ranking — Best Single-Split\nInformation Gain')

plt.suptitle('Feature Informativeness: Which Feature Creates the Best First Split?',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nDo Gini and Entropy agree on the ranking?')
rank_g = gain_df.sort_values('Best Gini Gain', ascending=False)['Feature'].tolist()
rank_e = gain_df.sort_values('Best Info Gain', ascending=False)['Feature'].tolist()
print(f'Gini ranking:    {rank_g}')
print(f'Entropy ranking: {rank_e}')
print(f'Rankings identical: {rank_g == rank_e}')

In [ ]:
# ── 9. Multi-class classification (3 price tiers) — criterion comparison ──────
# Multi-class highlights subtle differences between Gini and Entropy more than binary.

X_tr3, X_te3, y_tr3, y_te3 = train_test_split(
    X, y_3cls, test_size=0.25, random_state=42, stratify=y_3cls
)

print('=== 3-Class Price Tier: Criterion Comparison ===')
print(f'{"Criterion":<12} {"Depth":<8} {"Train Acc":<12} {"Test Acc":<12} {"Leaves"}')
print('-' * 55)

for crit in ['gini', 'entropy']:
    for depth in [3, 5, 7, None]:
        clf = DecisionTreeClassifier(criterion=crit, max_depth=depth, random_state=42)
        clf.fit(X_tr3, y_tr3)
        tr_acc = accuracy_score(y_tr3, clf.predict(X_tr3))
        te_acc = accuracy_score(y_te3, clf.predict(X_te3))
        leaves = clf.get_n_leaves()
        d_label = str(depth) if depth else 'None'
        print(f'{crit:<12} {d_label:<8} {tr_acc*100:>8.1f}%    {te_acc*100:>8.1f}%    {leaves}')

print()
print('3-class Gini impurity max = {:.4f}  (1 - 1/3)'.format(1 - 1/3))
print('3-class Entropy max       = {:.4f}  (log2(3))'.format(np.log2(3)))

In [ ]:
# ── 10. When do Gini and Entropy disagree? — Constructed example ───────────────
# Construct a node where the two criteria prefer different splits.
# This requires a near-tie in gain where entropy's log sensitivity changes the order.

# Toy node: 40 samples, 3 classes
y_toy = np.array([0]*20 + [1]*15 + [2]*5)   # 50% / 37.5% / 12.5%

# Two candidate splits:
# Split A: left=[0]*18+[1]*2, right=[0]*2+[1]*13+[2]*5
# Split B: left=[0]*20+[1]*5, right=[1]*10+[2]*5
ya_left  = np.array([0]*18 + [1]*2)
ya_right = np.array([0]*2  + [1]*13 + [2]*5)
yb_left  = np.array([0]*20 + [1]*5)
yb_right = np.array([1]*10 + [2]*5)

gA = gini_gain(y_toy, ya_left, ya_right)
gB = gini_gain(y_toy, yb_left, yb_right)
iA = information_gain(y_toy, ya_left, ya_right)
iB = information_gain(y_toy, yb_left, yb_right)

print('=== Constructed Example: When Might Gini and Entropy Disagree? ===')
print(f'Parent: {np.bincount(y_toy)} samples (class 0 / 1 / 2)')
print()
print('Split A splits into [18×0, 2×1] and [2×0, 13×1, 5×2]')
print(f'  Gini gain = {gA:.5f}')
print(f'  Info gain = {iA:.5f}')
print(f'  Gini prefers A: {gA > gB} | Entropy prefers A: {iA > iB}')
print()
print('Split B splits into [20×0, 5×1] and [10×1, 5×2]')
print(f'  Gini gain = {gB:.5f}')
print(f'  Info gain = {iB:.5f}')
print(f'  Gini prefers B: {gB > gA} | Entropy prefers B: {iB > iA}')
print()
if (gA > gB) != (iA > iB):
    print('>>> DISAGREEMENT: Gini and Entropy prefer DIFFERENT splits! <<<')
else:
    print('Both agree on the same split (typical outcome).')
    print('Disagreements are rare and require crafted edge cases.')

In [ ]:
# ── 11. How sklearn picks candidate thresholds ────────────────────────────────
# sklearn uses MIDPOINTS between consecutive unique values, not the raw values.
# This is more numerically stable and matches the standard CART description.

# Demonstrate on a small example
sqft_small = np.sort(np.random.choice(sqft_vals, 10, replace=False))
midpoints  = (sqft_small[:-1] + sqft_small[1:]) / 2

fig, ax = plt.subplots(figsize=(12, 3))
ax.scatter(sqft_small, np.zeros_like(sqft_small),
           s=120, color='steelblue', zorder=3, label='Unique sqft values')
ax.scatter(midpoints, np.zeros_like(midpoints) + 0.02,
           s=80, color='darkorange', marker='|', linewidths=2,
           zorder=4, label='Midpoint thresholds (sklearn uses these)')

for mp in midpoints:
    ax.axvline(mp, color='darkorange', lw=0.8, alpha=0.4, ls='--')

ax.set_yticks([])
ax.set_xlabel('sqft')
ax.set_title('sklearn Threshold Strategy: Midpoints Between Consecutive Values\n'
             'For n samples, at most n-1 candidate thresholds per feature')
ax.legend()
plt.tight_layout()
plt.show()

print('Example sqft values:', sqft_small.astype(int))
print('Midpoints used     :', midpoints.astype(int))
print()
print('Time complexity per node: O(n × p × log n)')
print('  - n = samples at node')
print('  - p = number of features')
print('  - log n factor from sorting')

In [ ]:
# ── 12. Gain landscape for ALL features side by side ─────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for j, (fname, ax) in enumerate(zip(feature_names, axes)):
    col = X[:, j]
    thresholds_j = np.unique(col)
    gains_g = []
    gains_e = []

    for t in thresholds_j:
        lm = col <= t
        rm = ~lm
        if lm.sum() == 0 or rm.sum() == 0:
            gains_g.append(0); gains_e.append(0); continue
        gains_g.append(gini_gain(y_bin, y_bin[lm], y_bin[rm]))
        gains_e.append(information_gain(y_bin, y_bin[lm], y_bin[rm]))

    gains_g = np.array(gains_g)
    gains_e = np.array(gains_e)
    best_t  = thresholds_j[np.argmax(gains_g)]

    ax.plot(thresholds_j, gains_g, lw=1.5, color='steelblue', label='Gini gain')
    ax.plot(thresholds_j, gains_e, lw=1.5, color='darkorange', ls='--', label='Info gain')
    ax.axvline(best_t, color='black', lw=1.5, ls=':', label=f'Best t={best_t:.1f}')
    ax.set_title(f'{fname}  (max Gini gain={gains_g.max():.4f})', fontsize=10)
    ax.set_xlabel(fname)
    ax.set_ylabel('Gain')
    if j == 0:
        ax.legend(fontsize=8)

plt.suptitle('Gain Landscape for All 6 Features (binary: above/below median)\n'
             'The feature with the highest peak gain becomes the root split',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 13. Relationship between Gini gain and Information Gain ───────────────────
# For each feature and threshold, compute both gains — plot as scatter.
# If they are perfectly correlated they would always pick the same threshold.

all_gini_gains = []
all_info_gains = []

for j in range(X.shape[1]):
    col = X[:, j]
    for t in np.unique(col):
        lm = col <= t; rm = ~lm
        if lm.sum() < 5 or rm.sum() < 5:
            continue
        all_gini_gains.append(gini_gain(y_bin, y_bin[lm], y_bin[rm]))
        all_info_gains.append(information_gain(y_bin, y_bin[lm], y_bin[rm]))

all_gini_gains = np.array(all_gini_gains)
all_info_gains = np.array(all_info_gains)
corr = np.corrcoef(all_gini_gains, all_info_gains)[0, 1]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(all_gini_gains, all_info_gains, alpha=0.3, s=10, color='purple')
ax.set_xlabel('Gini Gain')
ax.set_ylabel('Information Gain (bits)')
ax.set_title(f'Gini Gain vs Information Gain — All (feature, threshold) pairs\n'
             f'Pearson r = {corr:.4f}  (near 1.0 → nearly always same ranking)')
plt.tight_layout()
plt.show()

print(f'Pearson correlation between Gini gain and Information Gain: {corr:.6f}')
print('Near 1.0: the two criteria almost always rank candidate splits identically.')
print('This explains why Gini and Entropy trees give near-identical accuracy.')

## Summary: What You've Learned

| Concept | Key Takeaway |
|---|---|
| Gini Impurity | $1 - \sum p_k^2$; probability of misclassifying a random pair; 0 = pure, max = 1-1/K |
| Entropy | $-\sum p_k \log_2 p_k$; measures uncertainty; more sensitive near 50/50 splits |
| Information Gain | Entropy reduction after a split; equivalent to mutual information between feature and label |
| Variance Reduction | For regression; equivalent to minimising weighted MSE |
| Gini vs Entropy | Highly correlated ($r > 0.99$); practically interchangeable; Gini is slightly faster |
| Impurity landscape | CART scans all thresholds; picks the one that maximises gain |
| Feature ranking | Best-single-split gain orders features by informativeness (proto-importance) |

**Next Up:** Notebook 3 introduces **ensemble methods** — why combining many imperfect trees (Bagging / Random Forests) produces far better predictions than any single tree.

## Self-Check Questions

Test your understanding before moving on.

---

**Q1.** A node has 100 samples: 98 class A, 2 class B. What is the Gini impurity? Is this a "good" or "bad" node?

<details>
<summary>Show Answer</summary>

Using the formula $\text{Gini} = 1 - p_A^2 - p_B^2$:

$p_A = 0.98,\quad p_B = 0.02$

$\text{Gini} = 1 - (0.98)^2 - (0.02)^2 = 1 - 0.9604 - 0.0004 = 0.0392$

This is a **very good (nearly pure) node**. With Gini = 0.039 (out of a maximum of 0.5 for binary), the node is almost entirely class A. A decision tree would be happy to stop here — the majority-class prediction "class A" is correct 98% of the time. The node only needs to be split further if there are enough samples and a feature that separates the 2 class-B outliers. In many cases the tree would leave this as a leaf.

</details>

---

**Q2.** Gini and Entropy almost always select the same split. When might they **disagree**?

<details>
<summary>Show Answer</summary>

Disagreements are rare but can occur in two main situations:

1. **Near-tie situations in multi-class problems**: when two candidate splits produce almost identical gains, the curvature difference between Gini ($p^2$) and Entropy ($p \log p$) can tip the balance. Entropy's greater sensitivity near $p = 0.5$ (due to the logarithm) means it sometimes favours splits that produce more balanced children, while Gini may prefer splits that create one very pure child and one impure one.

2. **Highly imbalanced multi-class distributions**: when one class heavily dominates, the log term in entropy amplifies differences between the rare classes, while the squared term in Gini is less sensitive. This can lead to different threshold or even different feature selections.

In practice on most real datasets (especially binary classification), the Pearson correlation between Gini gain and Information Gain across all candidate splits exceeds 0.99, so the two criteria are functionally equivalent. sklearn's default of Gini is chosen purely for computational speed (no logarithm required).

</details>

---

**Q3.** For regression, we minimise **variance** rather than Gini. Why doesn't Gini apply to regression problems?

<details>
<summary>Show Answer</summary>

Gini impurity is defined using **class probabilities** $p_k$ — the fraction of samples belonging to each discrete class. In regression, the target $y$ is **continuous**: there are no discrete classes, so there are no class proportions to compute Gini from.

Instead, we need a measure of **spread** within a group. The natural choice is variance (or equivalently MSE), because the regression tree's leaf prediction is the **mean** of all samples in the leaf. The prediction error for any sample $i$ in that leaf is $(y_i - \bar{y})$, and the average squared error is exactly $\text{Var}(y)$.

So variance reduction is the regression analogue of Gini reduction: both measure how much *disorder* (heterogeneity) in the target variable is removed by the split. They are conceptually the same idea applied to different types of targets (discrete vs continuous).

</details>